# VHAGAR. Phase 0, Step 1: First Light

**Goal: get one real GOES fire detection and one real VIIRS detection into the same event object.**

Everything in VHAGAR so far is apparatus, physics, evaluation machinery, sensor registry. Nothing has touched a real byte. This notebook is where that changes.

Expect it to surface problems. That is the point.

Runs in Google Colab or any machine with network. GOES needs **no credentials** (NOAA's buckets are public and not requester-pays). FIRMS needs a free key from https://firms.modaps.eosdis.nasa.gov/api/map_key/

## 1. Install

In [ ]:
# In Colab: upload the VHAGAR folder to Drive, or clone it, then point this at it.
import os, sys, pathlib

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    VHAGAR = '/content/drive/MyDrive/VHAGAR'   # <-- adjust to where you put it
else:
    VHAGAR = str(pathlib.Path.cwd().parent)

%pip install -q -e "{VHAGAR}"
%pip install -q s3fs xarray h5netcdf pyproj

import vhagar
print('vhagar', vhagar.__version__)

## 2. Where and when

Pick somewhere with active fire. If you get nothing, that is a normal outcome, widen the box or the window, or check a fire map first (https://firms.modaps.eosdis.nasa.gov/map/).

**Which satellite:** GOES-19 is GOES-East (75.2°W), GOES-18 is GOES-West (137°W). For the western US, GOES-18 has the better viewing geometry, and viewing geometry matters more than you might think, because FRP scales with pixel area and with 1/transmittance, both of which degrade sharply off nadir.

In [ ]:
from datetime import datetime, timedelta, timezone

BBOX = (-124.0, 36.0, -118.0, 42.0)   # west, south, east, north, northern California
SATELLITE = 18                        # 18 = GOES-West, better geometry for the western US
HOURS = 6
REGION = 'conus'

os.environ.setdefault('FIRMS_MAP_KEY', '')   # <-- paste your free key here

end = datetime.now(timezone.utc)
start = end - timedelta(hours=HOURS)
print(f'{start:%Y-%m-%d %H:%M} to {end:%H:%M} UTC')

## 3. What is actually flying today

Not decoration. S-NPP data delivery ceases **2026-11-01 13:00 UTC**, which halves the VIIRS cadence over CONUS from ~25 to ~50 minutes. The registry knows, so your pipeline changes behaviour on that date rather than silently opening a hole.

In [ ]:
from datetime import date
from vhagar.io.sensors import coverage_report

print(coverage_report(date.today()))
print()
print(coverage_report(date(2026, 12, 1)))

## 4. List GOES granules

CONUS domain scans every 5 minutes, so a 6-hour window is ~72 granules.

In [ ]:
from vhagar.io.goes_reader import list_fdc_granules

keys = list_fdc_granules(SATELLITE, start, end, domain='C')
print(f'{len(keys)} granules')
for k in keys[-3:]:
    print(' ', k.rsplit('/', 1)[-1])

## 5. Read one granule

The bbox is converted to **scan-angle limits before decoding**, so this reads a small slice rather than converting the whole grid to lat/lon. That is the difference between a fast loop and an unusable one.

Look at the mask breakdown. Codes 10-15 are unfiltered detections; 30-35 are the same categories after the 12-hour temporal filter. Keeping only one throws away either latency or precision.

In [ ]:
from vhagar.io.goes_reader import open_fdc, mask_summary, read_fdc_detections

g = open_fdc(keys[-1], SATELLITE, bbox=BBOX)
print('scan start      ', g.scan_start)
print('cropped grid    ', g.mask.shape)
print('view zenith     ', f'{g.view_zenith_deg.min():.1f} to {g.view_zenith_deg.max():.1f} deg')
print('true pixel area ', f'{g.true_pixel_area_m2.mean() / 1e6:.2f} km2 '
      f'(nominal 2 km = 4.00 km2)')
print('fire pixels     ', g.n_fire_pixels(), 'unfiltered,', g.n_fire_pixels(filtered=True), 'filtered')
print()
for k, v in sorted(mask_summary(g).items(), key=lambda kv: -kv[1]):
    print(f'  {k:<45} {v}')

## 6. What the atmosphere is doing to your FRP

This is the largest single correctable systematic in geostationary FRP, and it is deterministic. At 60° view zenith the correction is ~2.1×.

In [ ]:
import numpy as np
from vhagar.physics.atmosphere import transmittance_mir, frp_atmospheric_correction_factor

TCWV = 20.0   # kg/m2, replace with ERA5 or GFS for real work
vza = np.array([g.view_zenith_deg.min(), g.view_zenith_deg.mean(), g.view_zenith_deg.max()])
for z in vza:
    tau = float(transmittance_mir(TCWV, z))
    print(f'  view zenith {z:5.1f} deg   tau {tau:.3f}   FRP correction '
          f'x{float(frp_atmospheric_correction_factor(TCWV, z)):.2f}')

## 7. Read the whole window

In [ ]:
from vhagar.grid import REGION_CRS

CRS = REGION_CRS[REGION]
goes_dets = []
for i, key in enumerate(keys, 1):
    try:
        gr = open_fdc(key, SATELLITE, bbox=BBOX)
    except Exception as exc:
        print(f'  skip {key.rsplit("/", 1)[-1][:40]}: {type(exc).__name__}')
        continue
    goes_dets.extend(read_fdc_detections(gr, crs=CRS))
    if i % 12 == 0:
        print(f'  {i}/{len(keys)} granules, {len(goes_dets)} detections')

print(f'\nGOES detections: {len(goes_dets)}')

## 8. VIIRS from FIRMS

In [ ]:
from pyproj import Transformer
from vhagar.harmonize.fusion import Detection
from vhagar.io.firms import FirmsClient

viirs_dets = []
if os.environ.get('FIRMS_MAP_KEY'):
    tf = Transformer.from_crs('EPSG:4326', CRS, always_xy=True)
    client = FirmsClient()
    for source in ('viirs_noaa21_nrt', 'viirs_noaa20_nrt'):
        try:
            recs = [r for r in client.area(source, BBOX, day_range=1) if r.acq_datetime >= start]
        except Exception as exc:
            print(f'  {source}: {type(exc).__name__}: {exc}')
            continue
        print(f'  {source:<22} {len(recs)} detections')
        for r in recs:
            x, y = tf.transform(r.longitude, r.latitude)
            viirs_dets.append(Detection(
                sensor='viirs', x=float(x), y=float(y), when=r.acq_datetime,
                frp_mw=r.frp, bt_mir_k=r.brightness, bt_tir_k=r.bright_t31,
                confidence={'l': 0.25, 'n': 0.6, 'h': 0.9}.get(str(r.confidence)[:1].lower(), 0.5)))
else:
    print('  FIRMS_MAP_KEY not set. GOES-only run.')

print(f'VIIRS detections: {len(viirs_dets)}')

## 9. Fuse

`extra_tolerance_m=2000` is the parallax allowance. Naive nearest-pixel matching between GOES and VIIRS reports **26-36 % apparent false alarms**; a 3×3 buffer drops that to **7-15 %**. That difference is geometry, not model quality.

In [ ]:
from vhagar.harmonize.fusion import cluster_detections, event_features

events = cluster_detections(goes_dets + viirs_dets, max_gap_hours=12.0, extra_tolerance_m=2000.0)
multi = [e for e in events if len(e.sensors) > 1]
print(f'{len(goes_dets) + len(viirs_dets)} detections -> {len(events)} events')
print(f'multi-sensor confirmed: {len(multi)} ({100 * len(multi) / max(len(events), 1):.0f}%)')
print('Single-sensor events are your false-alarm suspect pool.\n')

import pandas as pd
rows = []
for e in sorted(events, key=lambda e: len(e.detections), reverse=True)[:15]:
    f = event_features(e)
    rows.append({'event': e.event_id, 'det': len(e.detections), 'sensors': len(e.sensors),
                 'hours': round(e.duration_h, 1), 'peak_frp_mw': f['peak_frp_mw'],
                 'growth_mw_h': f['frp_growth_mw_per_h']})
pd.DataFrame(rows)

## 10. How far apart are GOES and VIIRS, really?

This is the plot that justifies the tolerance. If most separations sit inside 2 km, the parallax allowance is doing its job. If they don't, look at view zenith angle before touching the tolerance.

In [ ]:
import matplotlib.pyplot as plt

if goes_dets and viirs_dets:
    gx = np.array([d.x for d in goes_dets]); gy = np.array([d.y for d in goes_dets])
    seps = np.array([np.hypot(gx - d.x, gy - d.y).min() for d in viirs_dets])

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(seps / 1000, bins=40, color='#4C6EF5', edgecolor='white', linewidth=0.5)
    ax.axvline(2.0, color='#E8590C', linestyle='--', linewidth=1.5,
               label='2 km parallax tolerance')
    ax.set_xlabel('distance to nearest GOES detection (km)')
    ax.set_ylabel('VIIRS detections')
    ax.set_title('GEO / LEO separation')
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False)
    plt.tight_layout(); plt.show()

    for q in (50, 75, 90, 95):
        print(f'  p{q:<3} {np.percentile(seps, q):>8,.0f} m')
    print(f'\n  within 2 km: {100 * np.mean(seps <= 2000):.0f}%')
else:
    print('Need both sensors for this plot.')

## 11. Map

In [ ]:
back = Transformer.from_crs(CRS, 'EPSG:4326', always_xy=True)
fig, ax = plt.subplots(figsize=(8, 8))
for dets, colour, label, size in (
    (goes_dets, '#E8590C', f'GOES-{SATELLITE} (2 km)', 30),
    (viirs_dets, '#1C7ED6', 'VIIRS (375 m)', 12),
):
    if not dets:
        continue
    lon, lat = back.transform([d.x for d in dets], [d.y for d in dets])
    ax.scatter(lon, lat, s=size, c=colour, alpha=0.55, label=label, edgecolors='none')
ax.set_xlim(BBOX[0], BBOX[2]); ax.set_ylim(BBOX[1], BBOX[3])
ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
ax.set_title(f'Detections, last {HOURS} h')
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## What to look at before moving on

1. **Mask breakdown.** What fraction of GOES detections are low-probability (code 15)? That is your precision/recall dial and you set it, not the algorithm.
2. **GEO/LEO separation.** Does the 2 km tolerance cover the bulk of the distribution?
3. **Single-sensor events.** These are your false-alarm suspects and the population your Stage-2 persistence features exist to classify.
4. **View zenith range.** If it exceeds ~60°, your uncorrected FRP is low by more than a factor of two.

Next: `docs/07_PHASE0.md` step 2, the tile archive backfill, which is the long pole and should start running in the background while you do everything else.